In [13]:
import torch
print("GPU available:", torch.cuda.is_available())

GPU available: False


In [14]:
!pip install torchvision opencv-python seaborn -q

In [15]:
import os, cv2, torch, numpy as np, random
import torchvision.models as tv_models
import torchvision.transforms as transforms
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.layers import (Input, LSTM, GRU, SimpleRNN, Dense,
    Dropout, GlobalAveragePooling1D, Attention, Concatenate, LayerNormalization,
    MultiHeadAttention, Bidirectional)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings; warnings.filterwarnings("ignore")
print("TF:", tf.__version__, " | Torch:", torch.__version__)

TF: 2.19.0  | Torch: 2.9.0+cpu


In [16]:
dataset_root = "/kaggle/input/datasets/avanthikasuraj/ucf101-20/ucf101 dataset"

train_path = os.path.join(dataset_root, "train")
val_path   = os.path.join(dataset_root, "val")
test_path  = os.path.join(dataset_root, "test")

In [17]:
classes = sorted([d for d in os.listdir(train_path) 
           if os.path.isdir(os.path.join(train_path, d))])

label_map = {cls:i for i,cls in enumerate(classes)}
print(f"Found {len(classes)} classes:", classes)

Found 25 classes: ['ApplyEyeMakeup', 'ApplyLipstick', 'BabyCrawling', 'Basketball', 'BenchPress', 'Biking', 'BlowDryHair', 'Bowling', 'CliffDiving', 'CricketShot', 'SumoWrestling', 'Surfing', 'Swing', 'TableTennisShot', 'TaiChi', 'TennisSwing', 'ThrowDiscus', 'TrampolineJumping', 'Typing', 'UnevenBars', 'VolleyballSpiking', 'WalkingWithDog', 'WallPushups', 'WritingOnBoard', 'YoYo']


Pretrained Models

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── CNN-1: ResNet34 ──────────────────────────────────────────────────
resnet = tv_models.resnet34(pretrained=True)
resnet = torch.nn.Sequential(*list(resnet.children())[:-1])   # remove FC → output (512,)
resnet = resnet.to(device).eval()

# ── CNN-2: MobileNetV2 ───────────────────────────────────────────────
mobilenet = tv_models.mobilenet_v2(pretrained=True)
mobilenet.classifier = torch.nn.Identity()                     # remove classifier → output (1280,)
mobilenet = mobilenet.to(device).eval()

print("ResNet34 feature dim   : 512")
print("MobileNetV2 feature dim: 1280")

ResNet34 feature dim   : 512
MobileNetV2 feature dim: 1280


Fine tuning 

In [19]:
# ── Fine-tune ResNet34: freeze all, unfreeze layer4 + fc ────────────
for param in resnet.parameters():
    param.requires_grad = False

# Unfreeze the last residual block (layer4 is index 7 in Sequential)
for param in list(resnet.children())[7].parameters():
    param.requires_grad = True

trainable_resnet = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total_resnet     = sum(p.numel() for p in resnet.parameters())
print(f"ResNet34  — trainable: {trainable_resnet:,} / {total_resnet:,}")

# ── Fine-tune MobileNetV2: freeze all, unfreeze last 3 InvertedResidual blocks
for param in mobilenet.parameters():
    param.requires_grad = False

for block in list(mobilenet.features.children())[-3:]:
    for param in block.parameters():
        param.requires_grad = True

trainable_mob = sum(p.numel() for p in mobilenet.parameters() if p.requires_grad)
total_mob     = sum(p.numel() for p in mobilenet.parameters())
print(f"MobileNetV2 — trainable: {trainable_mob:,} / {total_mob:,}")

ResNet34  — trainable: 13,114,368 / 21,284,672
MobileNetV2 — trainable: 1,206,080 / 2,223,872


In [20]:
# Training transform — spatial augmentations applied per frame
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),          # smaller = faster; still fine for ResNet
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet stats
                         std=[0.229, 0.224, 0.225])
])

# Val/Test transform — deterministic, no augmentation
eval_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

Extract Features

In [21]:
SEQ_LEN    = 30    
FRAME_STEP = 5     

def extract_features(video_path, transform, is_train=False):
    cap = cv2.VideoCapture(video_path)
    features = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % FRAME_STEP == 0:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            img = transform(frame_rgb).unsqueeze(0).to(device)

            with torch.no_grad():
                f_resnet    = resnet(img).squeeze().cpu().numpy()         # (512,)
                f_mobilenet = mobilenet(img).squeeze().cpu().numpy()      # (1280,)

            combined = np.concatenate([f_resnet, f_mobilenet])            # (1792,)
            features.append(combined)
        frame_idx += 1

    cap.release()
    return np.array(features)   # (T, 1792)

Temporal Sequence Creation

In [22]:
def make_fixed_length_sequence(features, seq_len=SEQ_LEN):
    T = len(features)
    if T == 0:
        return None
    if T >= seq_len:
        # uniformly sample seq_len indices
        indices = np.linspace(0, T - 1, seq_len, dtype=int)
        return features[indices]
    else:
        # pad with zeros at the end
        pad = np.zeros((seq_len - T, features.shape[1]))
        return np.vstack([features, pad])

Dataset Builder

In [23]:
def build_dataset(data_path, transform, is_train=False, desc=""):
    X, y = [], []
    for cls in classes:
        folder = os.path.join(data_path, cls)
        if not os.path.exists(folder):
            continue
        print(f"  Processing class: {cls}")
        for video_file in tqdm(os.listdir(folder), desc=cls, leave=False):
            video_path = os.path.join(folder, video_file)
            feats = extract_features(video_path, transform, is_train)
            if len(feats) < 5:          # skip very short / corrupt clips
                continue
            seq = make_fixed_length_sequence(feats, SEQ_LEN)
            if seq is None:
                continue
            X.append(seq)
            y.append(label_map[cls])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32)


print("SEQ_LEN:", SEQ_LEN, " | feature_dim: 1792")

SEQ_LEN: 30  | feature_dim: 1792


Build Train / Val / Test

In [ ]:
print("=" * 50)
print("Building TRAIN dataset ...")
X_train, y_train = build_dataset(train_path, train_transform, is_train=True)
print(f"Train: {X_train.shape}, labels: {y_train.shape}")

print("=" * 50)
print("Building VAL dataset ...")
X_val, y_val = build_dataset(val_path, eval_transform)
print(f"Val  : {X_val.shape}, labels: {y_val.shape}")

print("=" * 50)
print("Building TEST dataset ...")
X_test, y_test = build_dataset(test_path, eval_transform)
print(f"Test : {X_test.shape}, labels: {y_test.shape}")

Building TRAIN dataset ...
  Processing class: ApplyEyeMakeup


  Processing class: ApplyLipstick


  Processing class: BabyCrawling


  Processing class: Basketball


  Processing class: BenchPress


  Processing class: Biking


Biking:  38%|███▊      | 38/100 [01:51<02:57,  2.86s/it]

Embedding Layer: Project 1792-d → 256-d

In [ ]:
from tensorflow.keras.layers import TimeDistributed

def add_embedding_projection(inputs, embed_dim=256):
    x = TimeDistributed(Dense(embed_dim, activation="relu"))(inputs)
    x = TimeDistributed(LayerNormalization())(x)
    return x

FEATURE_DIM = X_train.shape[2]   # 1792
NUM_CLASSES  = len(classes)
EMBED_DIM    = 256
print(f"Feature dim: {FEATURE_DIM}, Embed dim: {EMBED_DIM}, Classes: {NUM_CLASSES}")

Training Callbacks

In [ ]:
def get_callbacks(model_name):
    return [
        EarlyStopping(monitor="val_accuracy", patience=7,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=3, min_lr=1e-6, verbose=1),
        ModelCheckpoint(f"/kaggle/working/{model_name}_best.keras",
                        monitor="val_accuracy", save_best_only=True, verbose=0)
    ]

RNN

In [ ]:
def build_rnn(lr=0.001):
    inp = Input(shape=(SEQ_LEN, FEATURE_DIM))
    x   = add_embedding_projection(inp, EMBED_DIM)          # (B, 30, 256)
    x   = SimpleRNN(128, return_sequences=True, dropout=0.3)(x)
    x   = SimpleRNN(64,  dropout=0.3)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.4)(x)
    out = Dense(NUM_CLASSES, activation="softmax")(x)
    m   = Model(inp, out, name="RNN")
    m.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

rnn_model = build_rnn()
rnn_model.summary()

LSTM

In [ ]:
def build_lstm(lr=0.001):
    inp = Input(shape=(SEQ_LEN, FEATURE_DIM))
    x   = add_embedding_projection(inp, EMBED_DIM)
    x   = LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)(x)
    x   = LSTM(64,  dropout=0.3)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.4)(x)
    out = Dense(NUM_CLASSES, activation="softmax")(x)
    m   = Model(inp, out, name="LSTM")
    m.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

lstm_model = build_lstm()
lstm_model.summary()

GRU

In [ ]:
def build_gru(lr=0.001):
    inp = Input(shape=(SEQ_LEN, FEATURE_DIM))
    x   = add_embedding_projection(inp, EMBED_DIM)
    x   = GRU(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.2)(x)
    x   = GRU(64,  dropout=0.3)(x)
    x   = Dense(64, activation="relu")(x)
    x   = Dropout(0.4)(x)
    out = Dense(NUM_CLASSES, activation="softmax")(x)
    m   = Model(inp, out, name="GRU")
    m.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

gru_model = build_gru()
gru_model.summary()

Bidirectional LSTM + Multi Head Attention

In [ ]:
def build_attention_model(lr=0.001):
    inp = Input(shape=(SEQ_LEN, FEATURE_DIM))
    x   = add_embedding_projection(inp, EMBED_DIM)                         # (B,30,256)

    # Bidirectional LSTM for richer temporal representation
    x   = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3))(x)  # (B,30,256)

    # Multi-Head Self-Attention over the temporal dimension
    attn_out = MultiHeadAttention(num_heads=4, key_dim=64)(x, x)           # (B,30,256)
    x        = LayerNormalization()(x + attn_out)                          # residual

    x   = GlobalAveragePooling1D()(x)                                      # (B,256)
    x   = Dense(128, activation="relu")(x)
    x   = Dropout(0.4)(x)
    out = Dense(NUM_CLASSES, activation="softmax")(x)

    m = Model(inp, out, name="BiLSTM_Attention")
    m.compile(optimizer=Adam(lr), loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

attention_model = build_attention_model()
attention_model.summary()

Train Models

In [ ]:
EPOCHS     = 40
BATCH_SIZE = 32

print("Training RNN ...")
history_rnn = rnn_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks("rnn"), verbose=1)

print("Training LSTM ...")
history_lstm = lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks("lstm"), verbose=1)

print("Training GRU ...")
history_gru = gru_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks("gru"), verbose=1)

print("Training BiLSTM+Attention ...")
history_attn = attention_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=get_callbacks("attention"), verbose=1)

Re-build datasets for a shorter seq_len to compare

In [ ]:
LEARNING_RATES = [0.001, 0.0005]
SEQ_LENGTHS    = [20, 30]          # two different temporal context windows

hp_results = []

for lr in LEARNING_RATES:
    for sl in SEQ_LENGTHS:
        print(f"--- lr={lr}  seq_len={sl} ---")

        # Truncate/pad existing sequences to sl (avoids re-extracting features)
        Xtr = X_train[:, :sl, :] if sl <= X_train.shape[1] else np.pad(
            X_train, ((0,0),(0, sl-X_train.shape[1]),(0,0)))
        Xv  = X_val[:,   :sl, :] if sl <= X_val.shape[1]   else np.pad(
            X_val,   ((0,0),(0, sl-X_val.shape[1]),  (0,0)))

        hp_inp = Input(shape=(sl, FEATURE_DIM))
        hx     = add_embedding_projection(hp_inp, 128)
        hx     = GRU(64, dropout=0.3)(hx)
        hx     = Dense(NUM_CLASSES, activation="softmax")(hx)
        hp_model = Model(hp_inp, hx)
        hp_model.compile(optimizer=Adam(lr),
                         loss="sparse_categorical_crossentropy",
                         metrics=["accuracy"])

        h = hp_model.fit(Xtr, y_train,
                         validation_data=(Xv, y_val),
                         epochs=15, batch_size=32, verbose=0,
                         callbacks=[EarlyStopping(patience=4,
                                     restore_best_weights=True)])
        best_val = max(h.history["val_accuracy"])
        hp_results.append({"lr": lr, "seq_len": sl, "val_acc": best_val})
        print(f"  best val_acc = {best_val:.4f}")

print("Hyperparameter sweep summary:")
for r in hp_results:
    print(f"  lr={r['lr']}  seq_len={r['seq_len']}  → val_acc={r['val_acc']:.4f}")

Evaluation on test set

In [ ]:
def evaluate_model(model, X_test, y_test, name):
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"{name:25s}  test_loss={loss:.4f}  test_acc={acc:.4f}")
    return acc

print("Test-set results:")
acc_rnn   = evaluate_model(rnn_model,       X_test, y_test, "RNN")
acc_lstm  = evaluate_model(lstm_model,      X_test, y_test, "LSTM")
acc_gru   = evaluate_model(gru_model,       X_test, y_test, "GRU")
acc_attn  = evaluate_model(attention_model, X_test, y_test, "BiLSTM+Attention")

Model Comparision & Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Bar chart: test accuracy ─────────────────────────────────────────
model_names = ["RNN", "LSTM", "GRU", "BiLSTM+Attn"]
test_accs   = [acc_rnn, acc_lstm, acc_gru, acc_attn]
colors      = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

bars = axes[0].bar(model_names, test_accs, color=colors, width=0.5)
axes[0].set_ylim(0, 1)
axes[0].set_title("Test Accuracy Comparison", fontsize=13)
axes[0].set_ylabel("Accuracy")
for bar, v in zip(bars, test_accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}",
                 ha="center", va="bottom", fontsize=10)

# ── Training curves: val_accuracy ────────────────────────────────────
for hist, name, color in zip(
        [history_rnn, history_lstm, history_gru, history_attn],
        model_names, colors):
    axes[1].plot(hist.history["val_accuracy"], label=name, color=color)

axes[1].set_title("Validation Accuracy over Epochs", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val Accuracy")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/model_comparison.png", dpi=150)
plt.show()

# ── Hyperparameter heatmap ────────────────────────────────────────────
import pandas as pd
hp_df = pd.DataFrame(hp_results)
pivot = hp_df.pivot(index="lr", columns="seq_len", values="val_acc")
plt.figure(figsize=(5, 3))
sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu")
plt.title("HP Search: val_acc (lr × seq_len)")
plt.tight_layout()
plt.savefig("/kaggle/working/hp_heatmap.png", dpi=150)
plt.show()

Confusion Matrix - BiLSTM + Attention

In [ ]:
y_pred = np.argmax(attention_model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted", fontsize=11)
plt.ylabel("Actual",    fontsize=11)
plt.title("Confusion Matrix — BiLSTM + Attention", fontsize=13)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("/kaggle/working/confusion_matrix.png", dpi=150)
plt.show()

Classification Report

In [ ]:
print("Classification Report (BiLSTM + Attention):")
print(classification_report(y_test, y_pred, target_names=classes))